In [1]:
from camel.agents import ChatAgent
from camel.types import ModelType
from datasets import load_dataset
from librarian.model import create_openai_model
from librarian.evaluation import SimpleQAGrader
from librarian.schema import LibrarianResponse
# from librarian.prompt import LIBRARIAN_PROMPT
from pprint import pprint

/home/yuan/.cache/pypoetry/virtualenvs/librarian-yhAyT7eh-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
librarian_results = list(load_dataset("json", data_files="../results/simpleqa/librarian_simpleqa_0.json")["train"])
cot_results = list(load_dataset("json", data_files="../results/simpleqa/cot_simpleqa_0.json")["train"])
plain_results = list(load_dataset("json", data_files="../results/simpleqa/plain_simpleqa_0.json")["train"])

In [3]:
incorrect_dataset = [dp for i, dp in enumerate(librarian_results) if dp["prediction"]["librarian"]["grade"] == "INCORRECT" and cot_results[i]["prediction"]["cot"]["grade"] == "CORRECT"]

In [34]:
prompt = \
"""You are a helpful assistant to answer a question in two steps.

Step 1: Recall and list all relevant facts to the question.

Step 2: Using only the facts from Step 1, perform a step-by-step reasoning process that leads to your final answer.  

Final Output Format:
'''
Retrieved Facts: [ ... ]
Step-by-Step Reasoning: [ ... ]
Answer: [ ... ]
'''
"""

librarian_agent = ChatAgent(model=create_openai_model(ModelType.GPT_4O), system_message=prompt)
evaluator = SimpleQAGrader(ChatAgent(model=create_openai_model(ModelType.GPT_4O)))

In [35]:
for idx in range(10):
    pprint({
        "problem": incorrect_dataset[idx]["problem"],
        "answer": incorrect_dataset[idx]["answer"]
    }, sort_dicts=False)

    r = librarian_agent.step(incorrect_dataset[idx]["problem"], response_format=LibrarianResponse)
    r = eval(r.msgs[0].content)

    pprint(r, sort_dicts=False)
    grade = evaluator.grade(incorrect_dataset[idx]["problem"], incorrect_dataset[idx]["answer"], r["answer"])
    print(eval(grade)["grade"])

{'problem': 'Who requested the Federal Aviation Administration (FAA) implement '
            'a 900 sq mi (2,300 km2) temporary flight restriction zone over '
            'the operations areas of the Deepwater Horizon?',
 'answer': 'The Coast Guard'}
{'knowledge': 'During the Deepwater Horizon oil spill, the United States Coast '
              'Guard requested that the Federal Aviation Administration (FAA) '
              'implement a temporary flight restriction zone over the '
              'operations areas. This restriction was to ensure the safety of '
              'aircraft involved in the response efforts and to prevent '
              'interference with the ongoing operations.',
 'reasoning': '1. Identify the event: The Deepwater Horizon oil spill was a '
              'significant environmental disaster that required extensive '
              'response efforts.\n'
              '2. Determine who would be responsible for coordinating airspace '
              'during such an op

In [13]:
print("Problem:", incorrect_dataset[idx]["problem"])
print("Answer:", incorrect_dataset[idx]["answer"])
print("Previous Knowledge:", incorrect_dataset[idx]["prediction"]["librarian"]["response"]["knowledge"])
print("Previous Reasoning:", incorrect_dataset[idx]["prediction"]["librarian"]["response"]["reasoning"])
print("Previous Prediction:", incorrect_dataset[idx]["prediction"]["librarian"]["response"]["answer"])


Problem: Who requested the Federal Aviation Administration (FAA) implement a 900 sq mi (2,300 km2) temporary flight restriction zone over the operations areas of the Deepwater Horizon?
Answer: The Coast Guard
Previous Knowledge: [
  {"source": "Wikipedia", "timestamp": "2023-10-01", "fact": "The Federal Aviation Administration (FAA) implemented a 900 sq mi (2,300 km2) temporary flight restriction zone over the operations areas of the Deepwater Horizon oil spill at the request of BP."}
]
Previous Reasoning: 1. The retrieved knowledge states that the FAA implemented the temporary flight restriction zone over the Deepwater Horizon area.
2. The request for this implementation came from BP, the company involved in the Deepwater Horizon operations.
3. This information directly answers the question of who requested the FAA to implement the flight restriction.

Answer: The request for the FAA to implement a 900 sq mi (2,300 km2) temporary flight restriction zone over the operations areas of th